In [7]:
# [Cell 1]데이터 로드
import os
# 가장 먼저 실행해야 함: GPU 비활성화 (CPU만 사용)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# CPU 모드 확인
print(f"현재 사용 장치: {'CPU' if len(tf.config.list_physical_devices('GPU')) == 0 else 'GPU'}")

# 데이터 로드
df = pd.read_csv('Train.csv')

# 노이즈 데이터 분리 (마침표 밀도 < 0.01)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['period_count'] = df['text'].apply(lambda x: str(x).count('.'))
df['period_density'] = df['period_count'] / df['word_count']

noise_df = df[df['period_density'] < 0.01].copy()
print(f"전체 노이즈 데이터 개수: {len(noise_df)}")
print(f"라벨 분포:\n{noise_df['label'].value_counts()}")

# 데이터 준비
sentences = noise_df['text'].astype(str).tolist()
labels = noise_df['label'].values

현재 사용 장치: GPU
전체 노이즈 데이터 개수: 13933
라벨 분포:
label
0.0    6978
1.0    6955
Name: count, dtype: int64


In [8]:
# [Cell 2] 텍스트 -> 숫자 변환 (Tokenization)

# 하이퍼파라미터 설정
VOCAB_SIZE = 20000  # 단어장 크기 (충분히 크게)
MAX_LENGTH = 200    # 문장 길이 제한 (앞부분만 봐도 판별 가능할 것)
TRUNC_TYPE = 'post'
PADDING_TYPE = 'post'
OOV_TOK = "<OOV>"

# 토크나이저 생성 (불용어 제거 안 함! 그게 힌트니까)
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOK)
tokenizer.fit_on_texts(sentences)

word_index = tokenizer.word_index
print(f"발견된 고유 단어 수: {len(word_index)}")

# 시퀀스로 변환
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)

# 학습/검증 세트 분리
X_train, X_val, y_train, y_val = train_test_split(padded, labels, test_size=0.1, random_state=42)

print(f"학습 데이터 형태: {X_train.shape}")
print(f"검증 데이터 형태: {X_val.shape}")

발견된 고유 단어 수: 2654
학습 데이터 형태: (12539, 200)
검증 데이터 형태: (1394, 200)


In [9]:
# [Cell 3] 모델 설계 및 컴파일

model = tf.keras.Sequential([
    # 1. 임베딩 레이어: 단어를 벡터로 변환 (패턴 학습)
    tf.keras.layers.Embedding(VOCAB_SIZE, 32, input_length=MAX_LENGTH),
    
    # 2. 풀링 레이어: 문장 전체를 하나의 벡터로 압축 (단어 주머니 효과)
    tf.keras.layers.GlobalAveragePooling1D(),
    
    # 3. 히든 레이어 (패턴 조합)
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dropout(0.5), # 과적합 방지
    
    # 4. 출력 레이어 (0 또는 1)
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/home/epistachio/miniconda3/envs/deeplearning/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-12-08 23:45:43.465742: W tensorflow/compiler/mlir/tools/kernel_gen/tf_gpu_runtime_wrappers.cc:40] 'cuModuleLoadData(&module, data)' failed with 'CUDA_ERROR_INVALID_PTX'

2025-12-08 23:45:43.465797: W tensorflow/compiler/mlir/tools/kernel_gen/tf_gpu_runtime_wrappers.cc:40] 'cuModuleGetFunction(&function, module, kernel_name)' failed with 'CUDA_ERROR_INVALID_HANDLE'

2025-12-08 23:45:43.465807: W tensorflow/core/framework/op_kernel.cc:1842] INTERNAL: 'cuLaunchKernel(function, gridX, gridY, gridZ, blockX, blockY, blockZ, 0, reinterpret_cast<CUstream>(stream), params, nullptr)' failed with 'CUDA_ERROR_INVALID_HANDLE'


InternalError: {{function_node __wrapped__Cast_device_/job:localhost/replica:0/task:0/device:GPU:0}} 'cuLaunchKernel(function, gridX, gridY, gridZ, blockX, blockY, blockZ, 0, reinterpret_cast<CUstream>(stream), params, nullptr)' failed with 'CUDA_ERROR_INVALID_HANDLE' [Op:Cast] name: 